# Tutorial 2: A dense BioModels sweep — and what it can and cannot detect (`MODEL1907260003`)

Estimated time: 40-60 minutes

## Prerequisites — install BEFORE running this notebook

This tutorial executes a **published BioModels SBML model** (Lever et al. 2014). Beyond the framework + tutorial infrastructure that all tutorials need, T2 specifically requires the **BioModels SBML simulator**:

| Need | Why | Install command |
|---|---|---|
| `bayesian-metamodeling` + `[tutorials]` | Framework runtime + jupyter/matplotlib/numpy. Same as every tutorial. | `pip install -e ".[tutorials]"` |
| **`libroadrunner` + `tellurium`** | **T2 only.** The BioModels worker uses `libroadrunner` to actually simulate the SBML model. **Without these, the two sweeps skip with a preflight banner — reading the model, building and validating the spec, and planning the design still teach.** | `pip install libroadrunner tellurium` *or* `pip install -e ".[biomodels]"` |
| Network access (first run) | Downloads `MODEL1907260003.xml` from EBI BioModels — about **40 KB, a second or two**. After that the SBML is cached. | n/a — or set `MM_BIOMODELS_OFFLINE=1` and pre-populate the cache |

> **Heads up: `libroadrunner` and `tellurium` are PyPI-only — they are NOT on conda-forge.** Even if you're using a conda env, install them with `pip` (pip works inside a conda env). If you have multiple Python environments, make sure to activate the one running this notebook's kernel before installing — the bootstrap cell below prints the kernel's env path so you can check.

**If you don't want to install `libroadrunner` right now**, that's fine: the bootstrap cell below detects this and prints a multi-line preflight banner explaining what to install, in which env, and which steps still teach. You can do T2 in two passes: first read-through without installing, then come back and install + re-execute when you want the actual SBML runs.

(See Tutorial 0 for the full framework-vs-tutorial-vs-T2-only dependency matrix.)

## Learning aims
- **Package aim:** drive a real SBML model through the typed spec workflow — `validate` → `plan` → `run` — from inside Jupyter, and find the results in the centralized `sweep_rows.csv`.
- **Scientific aim:** read a published model well enough to predict, *before* running it, which of its parameters your readout can detect — then **measure** that sensitivity instead of eyeballing a colormap.

## Success criteria
- You can say in one sentence what this model computes (a kinetic-proofreading ladder), without using "signalling" as a placeholder.
- You run **two** sweeps that differ by four lines of spec, and you can read the printed log-log slopes to say which parameter the readout responds to, and by how much.
- You can explain why a **flat** result here is a measurement rather than a bug — and name the evidence that tells the two apart.

## Why this tutorial matters

`MODEL1907260003` is a **published BioModels entry** — Lever, Maini, van der Merwe & Dushek, "Phenotypic models of T cell activation," *Nature Reviews Immunology* 2014 (PubMed 25145757). It is a phenotypic model of the kinetic-proofreading mechanism governing T-cell receptor / peptide-MHC interactions.

This is the moment you stop running toy programs and start running real published systems with the **same spec contract** you used in T1. The output channel changes (a 1-D scalar in T1 becomes a 2-D time-series here), the runtime changes (a libroadrunner SBML simulation rather than a toy `python` invocation), but the spec-driven workflow is identical — and you will see the exact lines that differ, printed side by side.

There is a second, harder lesson stacked on top. Running a published model is easy; **running the right experiment on it is not.** We sweep two of its parameters, `k_on` and `k_p`, over the same ±20% window. One of them barely moves the readout at all. Working out *in advance* which one, and why, is the whole difference between a parameter scan and an experiment.

## Step 1: Set up, read the model, then build the sweep spec

**You can skim the next cell.** It is environment plumbing: find the repo root, make sure the SBML is cached locally, and check that an SBML simulator is installed *in this kernel's environment* (largely what `bayesmm doctor` did for you in T0, plus one extra tier). None of it is the lesson — but do read the banner it prints, because it decides whether the two sweeps execute.

In [ ]:
# Cross-platform setup (Windows / macOS / Linux) — no shell, no PYTHONPATH prefix.
# Find the repo root so `src/` is importable, then load the shared tutorial helpers.
import importlib.util
import os
import sys
from pathlib import Path

_root = Path.cwd().resolve()
while not (_root / "src" / "bayesian_metamodeling").is_dir() and _root != _root.parent:
    _root = _root.parent
if str(_root / "src") not in sys.path:
    sys.path.insert(0, str(_root / "src"))

from bayesian_metamodeling.tutorial import bootstrap, run_mm_cli, run_tool  # noqa: F401

root = bootstrap()  # chdir to repo root + ensure src/ on sys.path (idempotent)
ROOT = root
print("Repo root:", root)

import json
import shutil

BASE_SPEC_PATH = root / "tutorials/specs/model.biomodels.quick.json"
BASE_SPEC = json.loads(BASE_SPEC_PATH.read_text())
BIOMODELS_ID = BASE_SPEC["model"]["artifact"]["biomodels_id"]

# Two sweeps, two stores. Keeping them apart means "the newest sweep under this
# root" is an unambiguous way to find each one's results later.
KON_STORE = "tmp/tutorials/biomodels_store_dense"
KP_STORE = "tmp/tutorials/biomodels_store_kp"

# Offline-friendly cache bootstrap. Two paths supported by the adapter:
# 1) `MM_BIOMODELS_OFFLINE=1` env var: refuse to hit the network. Use this in
#    sandboxed/restricted environments. The adapter raises an actionable
#    error pointing at the missing cache path + a `curl` recipe.
# 2) `model.artifact.local_sbml_path` on the spec: ship a sample SBML in-tree
#    and the adapter copies it into the cache instead of downloading.
OFFLINE = os.environ.get("MM_BIOMODELS_OFFLINE", "").strip().lower() in {"1", "true", "yes", "on"}
VENDORED_SBML = root / "examples/biomodels/MODEL1907260003.xml"


def _cache_path(store_root: str) -> Path:
    return root / store_root / "_cache" / "biomodels" / f"{BIOMODELS_ID}.xml"


target_cache = _cache_path(KON_STORE)
CACHE_READY = target_cache.exists()
if not CACHE_READY:
    # Try seeding from a sibling tutorial cache the user may already have.
    for candidate in (root / "tmp/tutorials/biomodels_store/_cache/biomodels" / f"{BIOMODELS_ID}.xml",):
        if candidate.exists():
            target_cache.parent.mkdir(parents=True, exist_ok=True)
            shutil.copy2(candidate, target_cache)
            CACHE_READY = True
            print(f"Seeded local SBML cache: {candidate} -> {target_cache}")
            break

# Still no cache? Choose between fetching and the vendored copy.
#
# Fetching is the DEFAULT and is part of what T2 is teaching — pulling a real
# published model. But biomodels.org returns 403 to some cloud/CI address ranges
# while working fine from an ordinary connection, and without a fallback that
# turns into all 11/11 DOE points failing. So: probe first. A learner with a
# working connection still exercises the real download; a blocked or offline
# environment still gets to run the sweep and see the science.
FETCH_REACHABLE = None
if not CACHE_READY:
    if OFFLINE:
        _why = f"MM_BIOMODELS_OFFLINE={os.environ.get('MM_BIOMODELS_OFFLINE')!r}"
    else:
        import urllib.request

        _url = BASE_SPEC["model"]["artifact"].get("source_url")
        try:
            with urllib.request.urlopen(_url, timeout=20) as _resp:
                FETCH_REACHABLE = _resp.status == 200
            _why = None
        except Exception as _exc:  # noqa: BLE001 - any failure means "cannot fetch"
            FETCH_REACHABLE = False
            _why = f"{type(_exc).__name__}: {str(_exc)[:120]}"

    if (OFFLINE or FETCH_REACHABLE is False) and VENDORED_SBML.exists():
        target_cache.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(VENDORED_SBML, target_cache)
        CACHE_READY = True
        print("=" * 72)
        print("  BioModels download not used — seeding from the vendored SBML")
        print("=" * 72)
        print(f"  Reason : {_why}")
        print(f"  Source : {VENDORED_SBML.relative_to(root)}")
        print("  Only the download step is affected; the sweeps below run normally.")
        print("=" * 72)
        print()

# The second store needs the same SBML. Copy rather than re-download: the point
# of a cache is that a second experiment on the same model costs no network.
if CACHE_READY:
    _kp_cache = _cache_path(KP_STORE)
    if not _kp_cache.exists():
        _kp_cache.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(target_cache, _kp_cache)

# The file the sweeps will actually read. Used by the "read the model" cell below,
# which needs no simulator at all — just XML.
SBML_PATH = target_cache if target_cache.exists() else VENDORED_SBML

# Preflight: detect the SBML simulator. The sweeps invoke a worker script that
# uses `roadrunner` (libroadrunner) to run the SBML model. If it is not installed
# in this kernel's env, the run will fail no matter what — better to detect that
# BEFORE the run and say exactly what to do.
#
# Two-tier check:
#   (a) `LIBROADRUNNER_AVAILABLE` — `roadrunner` importable in THIS kernel.
#   (b) `LIBROADRUNNER_WORKER_OK` — a worker subprocess spawned with
#       `sys.executable` (which is what the framework's runner uses since the
#       28835b8 bug fix) can ALSO import `roadrunner`.
#
# Why both: the historical bug (Status.md post-mortem) was that the kernel had
# libroadrunner but workers used PATH-`python` (typically base conda) which did
# not. With (b) in place, even if the runner regresses or the user has a
# half-installed env, the preflight catches it instead of every DOE point
# silently failing.
import subprocess as _sp

LIBROADRUNNER_AVAILABLE = importlib.util.find_spec("roadrunner") is not None
LIBROADRUNNER_WORKER_OK = False
if LIBROADRUNNER_AVAILABLE:
    _check = _sp.run(
        [sys.executable, "-c", "import roadrunner"],
        capture_output=True, text=True, timeout=30,
    )
    LIBROADRUNNER_WORKER_OK = _check.returncode == 0

# Single verdict for whether the sweeps will actually execute.
RUN_WILL_EXECUTE = LIBROADRUNNER_AVAILABLE and LIBROADRUNNER_WORKER_OK and (CACHE_READY or not OFFLINE)

BANNER = "=" * 72
if RUN_WILL_EXECUTE:
    print()
    print(f"  Preflight OK: libroadrunner present, "
          f"{'cache ready' if CACHE_READY else 'first run will download SBML'}.")
elif LIBROADRUNNER_AVAILABLE and not LIBROADRUNNER_WORKER_OK:
    # Half-installed env: importable here, but a worker subprocess spawned with
    # sys.executable cannot import it. Means a non-standard env situation
    # (different python on PATH, broken venv layering, nbserver/kernel mismatch).
    print()
    print(BANNER)
    print("  PREFLIGHT: half-installed libroadrunner — the sweeps will be SKIPPED")
    print(BANNER)
    print(f"  Kernel env path  : {sys.prefix}")
    print("  This kernel's import of `roadrunner` works,")
    print("  BUT a worker spawned via `sys.executable` failed to import it.")
    print()
    print("  This usually means the env's site-packages is shadowed or the")
    print("  worker is hitting a different Python than the kernel. Try:")
    print("    1. `conda info --envs` — confirm the kernel env is what you expect.")
    print("    2. Recreate the env: `conda env remove -n <name> && conda env create -f environment.yml -n <name>`")
    print("    3. Then `pip install libroadrunner tellurium` inside the new env.")
    print("    4. Restart the kernel.")
    print(BANNER)
elif not LIBROADRUNNER_AVAILABLE:
    # Detect the active conda env name (or None) so we can tell the student WHICH
    # env this kernel uses — without hardcoding one. The install uses pip because
    # libroadrunner/tellurium are PyPI-only (not on conda-forge).
    _conda_env_name = os.environ.get("CONDA_DEFAULT_ENV")
    print()
    print(BANNER)
    print("  PREFLIGHT: BioModels SBML simulator missing — the sweeps will be SKIPPED")
    print(BANNER)
    print(f"  Kernel env path  : {sys.prefix}")
    if _conda_env_name:
        print(f"  Conda env name   : {_conda_env_name}")
    print("  Missing package  : 'roadrunner' (Python binding for libRoadRunner)")
    print()
    print("  Reading the model, building + validating the spec, and planning the")
    print("  design all still run — they need no simulator. Only the two sweeps")
    print("  and the analysis that depends on them are affected.")
    print()
    print("  HOW TO INSTALL — note: libroadrunner/tellurium are PyPI-only (NOT")
    print("  on conda-forge). Use pip even inside a conda env.")
    print()
    print("    # In a terminal, with THIS notebook's kernel env activated")
    if _conda_env_name:
        print(f"    # (your env: '{_conda_env_name}'; verify with `conda info --envs`):")
        print(f"    conda activate {_conda_env_name}")
    else:
        print("    # (activate it first — `conda info --envs` lists conda envs):")
    print("    pip install libroadrunner tellurium")
    print()
    print("    # Or, via the package's extras group (same effect):")
    print("    pip install -e \".[biomodels]\"")
    print()
    print("  After installing, RESTART the Jupyter kernel (cached imports won't")
    print("  pick up the new package) and re-run this notebook from the top.")
    print(BANNER)
elif OFFLINE and not CACHE_READY:
    print()
    print(BANNER)
    print("  OFFLINE MODE (MM_BIOMODELS_OFFLINE=1) and no cached SBML at:")
    print(f"    {target_cache}")
    print(BANNER)
    print("  The two sweeps will be SKIPPED. To execute them later, either:")
    print("    (a) unset MM_BIOMODELS_OFFLINE and re-run with network access, or")
    print("    (b) pre-populate the cache:")
    print(f"          curl -L <BioModels-URL> -o '{target_cache}'")
    print(BANNER)
else:
    print()
    print("  No local SBML cache found. The first run downloads it from BioModels")
    print("  (about 40 KB, a second or two). Later runs reuse the cache.")

### The model in one paragraph

Strip the COPASI annotations out of `MODEL1907260003.xml` and what is left is a **kinetic-proofreading ladder**. A free receptor `T` and a free ligand `P` bind to form the complex `C0` (rate law `k_on · P · T`). A bound complex is then modified one step at a time — `C0 → C1 → C2 → …` — every rung at the same rate `k_p`; and from *any* rung it can fall apart, at rate `k_off`. Only the top rung counts as signalling: the SBML defines `Activation_Normalised = C10 / T_Total`.

That geometry *is* proofreading. To signal, a complex must survive one coin flip per rung, each won with probability `k_p / (k_p + k_off)`. So the fraction that reaches the top is

```
survival = ( k_p / (k_p + k_off) ) ** N_rungs
```

— one power per rung. That exponent is how a T cell turns a modest difference in binding lifetime (`1/k_off`) into a decisive difference in response, and it is why proofreading discriminates on **`k_off`**, not on how *many* receptors are engaged.

Two facts about *this particular file* decide everything you are about to see. The next cell reads them straight out of the XML rather than asking you to take my word for it:

1. **How many rate laws each parameter appears in.** A parameter that enters one reaction out of twenty-two cannot do much.
2. **The initial condition.** If every complex starts already bound, there is no free-ligand pool left to titrate — the system *drains* the ladder rather than filling it, and the binding reaction only recaptures the trickle that falls off.

Predict both before you run the cell.

In [ ]:
# Read the model. Don't reason from its name, and don't reason from mine.
#
# Everything below is parsed out of the SBML the sweeps use: the cached copy if one
# exists, otherwise the vendored in-tree copy that seeds it.
# No simulator needed — an SBML file is just XML, and being able to open one and
# answer "what does this parameter touch?" is a skill worth ten heatmaps.
import xml.etree.ElementTree as ET

_tree = ET.parse(SBML_PATH)
_rootel = _tree.getroot()
NS = {"s": _rootel.tag.split("}")[0].lstrip("{")}
_model = _rootel.find("s:model", NS)

_params = {p.get("id"): p for p in _model.findall("s:listOfParameters/s:parameter", NS)}
_unitdefs = {u.get("id"): u.get("name") for u in _model.findall("s:listOfUnitDefinitions/s:unitDefinition", NS)}
_species = {
    s.get("id"): float(s.get("initialConcentration", "nan"))
    for s in _model.findall("s:listOfSpecies/s:species", NS)
}
_reactions = _model.findall("s:listOfReactions/s:reaction", NS)

# Which rate laws actually mention each parameter?
USES = {name: 0 for name in ("k_on", "k_off", "k_p")}
for _rxn in _reactions:
    _law = _rxn.find("s:kineticLaw", NS)
    if _law is None:
        continue
    _symbols = {c.text.strip() for c in _law.iter() if c.tag.endswith("}ci") and c.text}
    for _name in USES:
        if _name in _symbols:
            USES[_name] += 1

_initial_nonzero = {k: v for k, v in _species.items() if v}

print(f"SBML file : {SBML_PATH.relative_to(root)}  ({SBML_PATH.stat().st_size:,} bytes)")
print(f"model     : {_model.get('name')}")
print(f"species   : {len(_species)}   reactions: {len(_reactions)}")
print(f"nonzero at t=0: {_initial_nonzero}   (every other species starts at 0)")
print()
print(f"{'parameter':<10}{'value':>12}{'declared units':>18}{'rate laws it enters':>22}")
print("-" * 62)
for _name in ("k_on", "k_off", "k_p"):
    _p = _params[_name]
    _u = _p.get("units")
    _resolved = _unitdefs.get(_u, _u)
    print(f"{_name:<10}{float(_p.get('value')):>12g}{_u + ' = ' + str(_resolved):>18}"
          f"{str(USES[_name]) + ' / ' + str(len(_reactions)):>22}")

K_ON_DEFAULT = float(_params["k_on"].get("value"))
K_OFF_DEFAULT = float(_params["k_off"].get("value"))
K_P_DEFAULT = float(_params["k_p"].get("value"))
T_TOTAL = float(_params["T_Total"].get("value"))
N_RUNGS = USES["k_p"]          # one k_p reaction per rung: C0->C1, ..., C9->C10
TOP_RUNG = f"C{N_RUNGS}"

print()
print(f"ladder    : C0 -> ... -> {TOP_RUNG}  ({N_RUNGS} rungs at k_p; "
      f"{USES['k_off']} escape routes at k_off)")
print(f"readout   : Activation_Normalised = {TOP_RUNG} / T_Total, T_Total = {T_TOTAL:g}")
print("            (an SBML *assignment rule*, not a species — see the checkpoint at the end")
print("             for why that means it never reaches sweep_rows.csv)")

SURVIVAL = (K_P_DEFAULT / (K_P_DEFAULT + K_OFF_DEFAULT)) ** N_RUNGS
CLIMB_TIME = N_RUNGS / K_P_DEFAULT
print()
print(f"proofreading survival = (k_p/(k_p+k_off))**{N_RUNGS} = {SURVIVAL:.4f}")
print(f"  -> of the {_species['C0']:.0f} complexes present at t=0, about "
      f"{SURVIVAL * _species['C0']:.0f} should reach {TOP_RUNG} at least once.")
print(f"  k_off is {K_P_DEFAULT / K_OFF_DEFAULT:.0f}x smaller than k_p here, so almost")
print("  everything survives: this parameter set is a strong agonist, sitting on the")
print("  flat top of the proofreading curve where discrimination has already been won.")
print()
print(f"mean time to climb all {N_RUNGS} rungs = N/k_p = {CLIMB_TIME:.1f} s")
print(f"  -> the shipped spec stops at t1={BASE_SPEC['io_schema']['time_grid']['t1']:g} s, which is "
      f"{BASE_SPEC['io_schema']['time_grid']['t1'] / CLIMB_TIME:.2f} of one climb.")
print("     We widen that window below. When you stop watching is a modeling decision.")

**What `k_on` is — in this file, not in general**

`k_on` is the association rate constant of the single binding reaction `P + T → C0`, with rate law `k_on · P · T`. Larger `k_on` means a free pMHC and a free TCR recapture each other faster. The cell above told you it appears in **1 of the 22 rate laws**, and that at `t = 0` the only nonzero species is `C0`: everything starts bound, so `P` and `T` — both of `k_on`'s substrates — start at zero.

**Its declared units are not usable as stated.** The SBML declares `<parameter id="k_on" units="unit_0" value="0.0001">`, and `unit_0` resolves to `1/s`. But the rate law is *bimolecular*: `k_on · P · T` can only be molecules-per-second if `k_on` carries units of per-molecule-per-second. A first-order constant like `k_p` genuinely is `1/s`; `k_on` is not, and the file says so anyway. The tutorial's own spec used to claim `1/(M·s)`, which is wrong in the other direction — the species carry `initialConcentration="30000"`, a per-cell **molecule count**, not a molarity.

**The habit worth stealing.** Before you sweep a published parameter, check its value against literature. Measured protein-protein association rates are of order 10⁴–10⁶ M⁻¹s⁻¹. A `k_on` of 10⁻⁴ M⁻¹s⁻¹ would sit about ten orders of magnitude below anything ever measured — which is not a claim about biology, it is a tell that the model is written in molecule counts. **Declared units in an SBML are documentation, and documentation drifts. The rate laws and the initial conditions are the source of truth.**

We sweep `k_on` on an 11-point grid spanning the published value ±20%.

### The spec is the same contract you used in T1

T1 ran a toy Python program. This runs a published SBML model on a different simulator with a different output shape. If the framework's promise is real, the two specs should be the **same object with a few slots filled differently** — not two different file formats.

The next cell builds the dense sweep spec and then prints the toy spec beside it, slot by slot, so you can see exactly which lines carry the difference and what the adapter does with each one. Two of the changes we make ourselves, for reasons the previous cells earned:

- **the observation window** — the shipped spec watches only the first fraction of one climb up the ladder, so we widen `io_schema.time_grid`;
- **the units string** — so the spec says what the file says.

In [ ]:
import numpy as np

# ---- build the dense k_on sweep spec, starting from the shipped 3-point one ----
SPEC_KON = json.loads(json.dumps(BASE_SPEC))  # deep copy; never mutate the shipped file

KON_GRID = [float(f"{v:.8g}") for v in np.linspace(0.8 * K_ON_DEFAULT, 1.2 * K_ON_DEFAULT, 11)]
SPEC_KON["design"]["grid"]["k_on"] = KON_GRID
SPEC_KON["io_schema"]["inputs"][0]["support"] = [KON_GRID[0], KON_GRID[-1]]
SPEC_KON["storage"]["root"] = KON_STORE

# Widen the observation window to ~3 climbs of the ladder, on a 2 s grid. The
# shipped 0-20 s window catches only the leading edge of the response; with this
# one you can watch C10 fill, peak, and start to drain.
T_WINDOW = float(round(3 * CLIMB_TIME / 10.0) * 10)          # 120 s here
N_TIME = int(T_WINDOW / 2.0) + 1                              # 2 s spacing
SPEC_KON["io_schema"]["time_grid"] = {"t0": 0.0, "t1": T_WINDOW, "n_points": N_TIME}

DENSE_SPEC_PATH = root / "tmp/tutorials/specs/model.biomodels.tutorial2.dense.json"
DENSE_SPEC_PATH.parent.mkdir(parents=True, exist_ok=True)
DENSE_SPEC_PATH.write_text(json.dumps(SPEC_KON, indent=2, sort_keys=True))
DENSE_SPEC_REL = str(DENSE_SPEC_PATH.relative_to(root))

# ---- what actually differs from the toy spec ----
TOY = json.loads((root / "tutorials/specs/model.toy.grid.json").read_text())

print("Top-level keys — toy spec vs BioModels spec:")
print("  toy      :", sorted(TOY))
print("  biomodels:", sorted(SPEC_KON))
print("  identical:", sorted(TOY) == sorted(SPEC_KON))
print()


def _slot(label, toy_value, bio_value, note):
    print(f"{label}")
    print(f"    toy       : {toy_value}")
    print(f"    biomodels : {bio_value}")
    print(f"    -> {note}")
    print()


_slot(
    "model.artifact.type",
    TOY["model"]["artifact"]["type"],
    SPEC_KON["model"]["artifact"]["type"],
    "picks the adapter: python_cli materializes a command line; biomodels_sbml "
    "fetches/caches an SBML and calls the roadrunner worker.",
)
_slot(
    "model.artifact.<source>",
    f'entrypoint={TOY["model"]["artifact"]["entrypoint"]}',
    f'biomodels_id={SPEC_KON["model"]["artifact"]["biomodels_id"]!r}',
    "where the model comes from. The BioModels id (plus source_url) is resolved "
    "once into <storage.root>/_cache/biomodels/ and reused by every DOE point.",
)
_slot(
    "adapter.input_mapping[0].to",
    TOY["adapter"]["input_mapping"][0]["to"],
    SPEC_KON["adapter"]["input_mapping"][0]["to"],
    "how a design point reaches the model. cli_arg appends `--a <value>`; "
    "sbml_parameter becomes rr.setGlobalParameterByName('k_on', value) inside the worker.",
)
_slot(
    "adapter.output_mapping[0].from",
    TOY["adapter"]["output_mapping"][0]["from"],
    SPEC_KON["adapter"]["output_mapping"][0]["from"],
    "how results come back. `file` reads a path the model wrote; `generated` means "
    "the adapter itself produced the payload (here: the roadrunner trajectory).",
)
_slot(
    "io_schema.time_grid",
    TOY["io_schema"].get("time_grid", "(absent — the toy model returns a scalar pair)"),
    SPEC_KON["io_schema"]["time_grid"],
    "the observation window, handed to the worker as rr.simulate(t0, t1, n_points). "
    "This is an experimental design choice, not a formatting detail.",
)

print(f"units check — spec says io_schema.inputs[0].units = "
      f"{SPEC_KON['io_schema']['inputs'][0]['units']!r}")
print(f"              SBML says k_on units = {_params['k_on'].get('units')!r} "
      f"-> {_unitdefs.get(_params['k_on'].get('units'))!r}")
print()
print("Wrote", DENSE_SPEC_REL)
print("Dense k_on grid:", KON_GRID)

In [ ]:
# `_ =` keeps the CLI exit code out of the cell result; we check the artifact,
# not the return value, in Step 3.
_ = run_mm_cli("validate", DENSE_SPEC_REL)
print()
_ = run_mm_cli("plan", DENSE_SPEC_REL)

**What the plan is telling you.** `validate` type-checks the spec — it never touches the model. `plan` is the next separable step: it expands `design` into the **design matrix**, the concrete list of parameter settings that will be executed. Nothing has run yet, and that is the point — a design you can inspect before spending compute is a design you can criticize.

Each of these 11 planned points becomes **exactly one row** of `sweep_rows.csv`, with its inputs, its outputs, its status and its timing. That table is the framework's canonical artifact: `run` writes it, Step 6 plots it, and in T5 a *surrogate* is fitted to a table of exactly this shape. (T5 uses the toy sweep rather than this one, for a reason worth knowing now — see the checkpoint at the end.)

**Why plan a design at all, rather than just running a lot of points?** Here each point is one SBML integration and costs a fraction of a second, so the honest answer is: for *this* model, budget is not the constraint — you could run 10,000 points. Two other things are. First, a dense grid over one input is cheap, but grids grow as `n**d`: at 11 points per axis, three inputs is 1,331 runs and five is 161,051. That is what T4's `sobol` strategy exists for, and why it only starts to matter once you have more than one input — this spec has exactly one. Second, and more immediately: a sweep that varies the wrong parameter is 100% wasted no matter how cheap it is. Which is the subject of the next cell.

## Predict before you run

Commit to answers before executing. Write them down — the tables below will settle every one of them.

1. `k_on` enters **1 of 22** rate laws, and both of its substrates start at zero. Over a ±20% window, does the activation readout `C10/T_Total` move by more than 10%? More than 1%? At all?
2. `k_p` enters **10** rate laws and sets the speed of the whole ladder. At `t = 20 s` — about half a climb in — does `C10` move by more than a factor of two over the same ±20% window?
3. Now the same question at the **end** of the window, once the ladder has finished filling. Bigger, smaller, or the same?

Both sweeps print a **log-log slope**, `d ln(output) / d ln(input)`: the percent change in the output per percent change in the input, also called an elasticity. It is the right currency for "how sensitive is this", because it is dimensionless and comparable across parameters that have different units and different magnitudes. Predict the **sign** and the **order of magnitude** of that number for each (parameter, readout, time) combination before you look.

This is the same predict-then-check habit T1 built on the toy heatmaps, and it is doing more work here: on a toy model you can verify by arithmetic; on a published biological model your prediction is the only thing standing between a plausible-looking plot and a silently wrong one.

*(A cautionary note, since this notebook once got it wrong itself: an earlier version of this cell confidently predicted a saturating binding curve with a knee in the middle — "binding increases with `k_on` until the limiting species is depleted". Read the initial conditions again. There is no free-ligand pool in this file to deplete. The prediction was reasoning from the name of the parameter instead of from the model, and the heatmap it produced looked plausible enough that nobody noticed for months.)*

## Step 2: Execute the dense `k_on` sweep

In [ ]:
if not RUN_WILL_EXECUTE:
    # Already explained in the preflight banner from the bootstrap cell above.
    if not LIBROADRUNNER_AVAILABLE:
        reason = "libroadrunner missing in this kernel"
    elif OFFLINE and not CACHE_READY:
        reason = "MM_BIOMODELS_OFFLINE=1 and cache empty"
    else:
        reason = "preflight check failed"
    print(f"Sweep SKIPPED ({reason}) — see preflight banner above.")
    run_exit_code = -1
else:
    run_exit_code = run_mm_cli("run", DENSE_SPEC_REL, check=False)

## Step 3: Did it work? Ask the data, not the exit code

In [ ]:
# `bayesmm run` exits 1 if ANY point failed — one failure out of eleven and ten
# out of eleven look identical from the outside. So verify against the artifact.
import csv


def newest_sweep_rows(store_root: str):
    """(csv_path, rows) for the most recent sweep under a store root; (None, []) if none."""
    sweeps = sorted(
        (root / store_root / "sweeps").glob("*/sweep_rows.csv"),
        key=lambda p: p.stat().st_mtime,
    )
    if not sweeps:
        return None, []
    path = sweeps[-1]
    with open(path, newline="") as fh:
        return path, list(csv.DictReader(fh))


def report_sweep(store_root: str, expected: int) -> int:
    """Print a data-derived verdict; return the number of successful points."""
    path, rows = newest_sweep_rows(store_root)
    if path is None:
        print(f"  no sweep_rows.csv under {store_root}/sweeps/ — nothing ran.")
        return 0
    ok = [r for r in rows if r.get("status") == "success"]
    print(f"  newest sweep : {path.relative_to(root)}")
    print(f"  points       : {len(ok)} succeeded / {len(rows)} rows / {expected} planned")
    if len(ok) == len(rows) == expected:
        print("  verdict      : complete — every planned point produced output.")
    elif ok:
        print("  verdict      : partial — the analysis below uses the successful rows only.")
        print(f"                 per-point errors live in {path.parent / 'sweep_logs.jsonl'}")
    else:
        print("  verdict      : every point failed. Read sweep_logs.jsonl; the usual cause")
        print("                 is the worker subprocess landing in the wrong interpreter.")
    return len(ok)


print(f"CLI exit code from Step 2: {run_exit_code}")
print("  (-1 means the preflight skipped the run; 1 means at least one point failed,")
print("   which is NOT the same as 'the sweep is useless'.)")
print()
if RUN_WILL_EXECUTE:
    KON_OK = report_sweep(KON_STORE, len(KON_GRID))
else:
    # Deliberately do NOT read the store here: it may still hold a sweep from an
    # earlier session with a different spec, and analysing that would be exactly
    # the kind of quiet false success this notebook exists to warn about.
    KON_OK = 0
    print("  the sweep did not run in this kernel, so nothing is read from the store —")
    print("  a stale artifact from a previous session is not evidence about this one.")

if KON_OK == 0:
    LINE = "-" * 72
    print()
    print(LINE)
    print("  No usable sweep data. Two paths forward:")
    print(LINE)
    print()
    print("  PATH A - finish the notebook with what you already have:")
    print("    The spec was built, validated and planned above, and the model was read")
    print("    straight out of its SBML. The remaining cells will say 'no data' and")
    print("    you can move on to T3 with the plumbing lesson intact.")
    print()
    print("  PATH B - actually run the model:")
    print("    1. Install libroadrunner in THIS kernel's env (see the preflight banner).")
    print("    2. In a sandbox with no network, either set MM_BIOMODELS_OFFLINE=1 and")
    print("       pre-populate the cache, or set model.artifact.local_sbml_path on the")
    print("       spec. Both are documented in bayesian_metamodeling.adapters.biomodels_sbml.")
    print("    3. Restart the kernel and re-run from the top.")
    print()
    print(LINE)

## Step 4: What did `k_on` actually do?

A colormap is a poor instrument for a small effect: a 2% change and a 0.002% change look identical once both are squeezed into the same 256 colours. So before any figure, read the numbers.

The table below reports each swept value against three readouts at a fixed time: `C0` (the bottom rung — it *falls* from 30000), `C10` (the top rung — it *rises* from 0), and the model's own activation readout `C10/T_Total`. Then two summaries per readout:

- **relative spread**, `(max − min) / mean` across the sweep — how much the output moved, in its own units;
- **log-log slope**, `d ln(output) / d ln(input)` — how much it moved *per percent of input*, dimensionless and comparable between parameters.

We evaluate at `t = 20 s` rather than at the end of the window, for a reason you should be suspicious of until you check it: by the end of the window `C0` has decayed to a small fraction of a single molecule per cell, and a continuum ODE happily keeps reporting values long after the quantity it models has stopped existing. The cell prints both times so you can watch that happen rather than take it on trust.

In [ ]:
import numpy as np


def load_sweep(store_root: str, input_var: str):
    """(csv_path, records) where each record is (x, time_array, {species: array}).

    Only status='success' rows, sorted by the swept input value.
    """
    path, rows = newest_sweep_rows(store_root)
    records = []
    for r in rows:
        if r.get("status") != "success" or not r.get("time_series__rows__json"):
            continue
        table = json.loads(r["time_series__rows__json"])
        t = np.array([float(e["time"]) for e in table])
        series = {k: np.array([float(e[k]) for e in table]) for k in table[0] if k != "time"}
        records.append((float(r[input_var]), t, series))
    records.sort(key=lambda item: item[0])
    return path, records


def loglog_slope(xs, ys) -> float:
    """d ln(y) / d ln(x) across the swept range: percent out per percent in."""
    xs = np.asarray(xs, dtype=float)
    ys = np.asarray(ys, dtype=float)
    if ys[0] <= 0 or ys[-1] <= 0 or xs[0] <= 0 or xs[-1] <= 0:
        return float("nan")
    return float(np.log(ys[-1] / ys[0]) / np.log(xs[-1] / xs[0]))


T_EVAL = 20.0  # seconds; the shipped spec's whole window, and where C0 is still readable


def sensitivity(store_root: str, input_var: str, label: str):
    """Print the per-point table + the two summary statistics. Returns a stats dict."""
    path, recs = load_sweep(store_root, input_var)
    if not recs:
        print(f"[{label}] no successful sweep with plottable time-series under {store_root}.")
        return None
    t = recs[0][1]
    i_eval = int(np.argmin(np.abs(t - T_EVAL)))
    i_end = len(t) - 1
    print(f"[{label}] {path.relative_to(root)}   {len(recs)} points, "
          f"{len(t)} time samples, t in [{t[0]:g}, {t[-1]:g}] s")
    print(f"          evaluated at t = {t[i_eval]:g} s (index {i_eval})")
    print()
    print(f"{input_var:>12}{'C0':>16}{'C10':>16}{'C10/T_Total':>16}")
    print("-" * 60)
    for x, _, s in recs:
        print(f"{x:>12.6g}{s['C0'][i_eval]:>16.6f}{s['C10'][i_eval]:>16.6f}"
              f"{s['C10'][i_eval] / T_TOTAL:>16.8f}")
    xs = [r[0] for r in recs]
    stats = {}
    print()
    for readout in ("C0", "C10"):
        ys = [r[2][readout][i_eval] for r in recs]
        rel = (max(ys) - min(ys)) / abs(np.mean(ys))
        slope = loglog_slope(xs, ys)
        stats[readout] = {"rel_spread": rel, "slope": slope}
        print(f"  {readout:>4} @ t={t[i_eval]:g}s : relative spread {rel:12.4e}"
              f"     d ln {readout} / d ln {input_var} = {slope:>+13.4g}")
    ys_end = [r[2]["C10"][i_end] for r in recs]
    stats["C10_end"] = {
        "rel_spread": (max(ys_end) - min(ys_end)) / abs(np.mean(ys_end)),
        "slope": loglog_slope(xs, ys_end),
        "t": float(t[i_end]),
    }
    print(f"  {'C10':>4} @ t={t[i_end]:g}s: relative spread "
          f"{stats['C10_end']['rel_spread']:12.4e}"
          f"     d ln C10 / d ln {input_var} = {stats['C10_end']['slope']:>+13.4g}")
    c0_end = [r[2]["C0"][i_end] for r in recs]
    print(f"  (for scale: C0 at t={t[i_end]:g}s spans {min(c0_end):.3g} to {max(c0_end):.3g} "
          f"molecules per cell)")
    if max(c0_end) < 1.0:
        print("   -- that is less than ONE molecule per cell, so any slope computed there is")
        print("   arithmetic on numerical dust, not biology. A continuum ODE keeps producing")
        print("   numbers long after the quantity it models has stopped existing. Pick")
        print("   readouts, and times, where the model still means something.")
    peak = max(r[2]["C10"].max() for r in recs)
    stats["C10_peak_fraction"] = float(peak / T_TOTAL)
    print(f"  window check: C10 peaks at {peak / T_TOTAL:.3f} x T_Total inside this window")
    print(f"                (proofreading survival predicted {SURVIVAL:.3f} would reach the top)")
    return stats


if KON_OK:
    KON_STATS = sensitivity(KON_STORE, "k_on", "k_on sweep")
else:
    KON_STATS = None
    print("No k_on sweep data in this session, so there is no sensitivity to report.")
    print("This table is the payload of Steps 2-4; install libroadrunner and re-run to get it.")

if KON_STATS:
    s = KON_STATS["C10"]["slope"]
    print()
    print(f"  Read the slope as: a 1% increase in k_on changes C10 by {s:.2e} percent.")
    print(f"  Across the full +/-20% sweep the activation readout moved by "
          f"{KON_STATS['C10']['rel_spread'] * 100:.5f}%.")
    print("  That is not a bug, and the slope being small but NONZERO is the evidence:")
    print("  a parameter that never reached the model would give exactly 0.0, and a")
    print("  monotone trend across 11 points is not what a plumbing failure looks like.")
    print("  You have measured an insensitivity — a real, publishable kind of result,")
    print("  and one you can only trust because you swept densely enough to see it.")

## Step 5: Four lines, a different experiment

Nothing about the model changed; the *question* did. To sweep `k_p` instead of `k_on` we edit four slots of the spec — the input declaration, the design grid, the adapter's input mapping, and the storage root — and leave the other twenty untouched. That is what "the spec is the experiment" means in practice, and it is the reason a typed spec beats a script: the diff is the hypothesis.

Two things to notice while it runs:

- `k_p` is genuinely a first-order rate constant, so the SBML's declared `1/s` is *correct* for it — unlike `k_on`, whose bimolecular rate law makes the same declaration unusable. Same file, same unit definition, one right and one wrong.
- We keep the window, the grid density, the seed and the readouts identical. A comparison between two sweeps is only worth making if the only thing that differs is the thing you are comparing.

In [ ]:
# ---- four slots, one new experiment ----
SPEC_KP = json.loads(json.dumps(SPEC_KON))  # start from the k_on spec: same window, same density

KP_GRID = [float(f"{v:.8g}") for v in np.linspace(0.8 * K_P_DEFAULT, 1.2 * K_P_DEFAULT, len(KON_GRID))]
SPEC_KP["io_schema"]["inputs"] = [
    {"name": "k_p", "type": "float", "units": "1/s", "support": [KP_GRID[0], KP_GRID[-1]]}
]
SPEC_KP["design"]["grid"] = {"k_p": KP_GRID}
SPEC_KP["adapter"]["input_mapping"] = [
    {"var": "k_p", "to": {"kind": "sbml_parameter", "key": "k_p"}}
]
SPEC_KP["storage"]["root"] = KP_STORE

KP_SPEC_PATH = root / "tmp/tutorials/specs/model.biomodels.tutorial2.kp.json"
KP_SPEC_PATH.write_text(json.dumps(SPEC_KP, indent=2, sort_keys=True))
KP_SPEC_REL = str(KP_SPEC_PATH.relative_to(root))

print("What changed, relative to the k_on spec:")
for _path_str in ("io_schema.inputs", "design.grid", "adapter.input_mapping", "storage.root"):
    _a, _b = SPEC_KON, SPEC_KP
    for _key in _path_str.split("."):
        _a, _b = _a[_key], _b[_key]
    print(f"  {_path_str}")
    for _tag, _val in (("k_on", _a), ("k_p ", _b)):
        _txt = json.dumps(_val)
        print(f"      {_tag} spec: {_txt[:150]}{'...' if len(_txt) > 150 else ''}")
print()
_unchanged = [k for k in SPEC_KON if json.dumps(SPEC_KON[k], sort_keys=True) == json.dumps(SPEC_KP[k], sort_keys=True)]
print("Top-level sections byte-identical between the two experiments:", sorted(_unchanged))
print()

_kon_cache = root / KON_STORE / "_cache" / "biomodels" / f"{BIOMODELS_ID}.xml"
_kp_cache = root / KP_STORE / "_cache" / "biomodels" / f"{BIOMODELS_ID}.xml"
if _kon_cache.exists() and not _kp_cache.exists():
    _kp_cache.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(_kon_cache, _kp_cache)
    print("Reused the SBML already cached by the first sweep - a second experiment\n           on the same model costs no network.")
    print()

if not RUN_WILL_EXECUTE:
    print("Contrast sweep SKIPPED (no SBML simulator) — see the preflight banner above.")
    kp_exit_code = -1
    KP_OK = 0
else:
    run_mm_cli("validate", KP_SPEC_REL)
    print()
    kp_exit_code = run_mm_cli("run", KP_SPEC_REL, check=False)
    print()
    KP_OK = report_sweep(KP_STORE, len(KP_GRID))

In [ ]:
if KP_OK:
    KP_STATS = sensitivity(KP_STORE, "k_p", "k_p sweep")
else:
    KP_STATS = None
    print("No k_p sweep data in this session — the contrast that makes Step 5 a lesson")
    print("needs both sweeps to have run.")

if KON_STATS and KP_STATS:
    print()
    print("=" * 78)
    print("  Same model. Same +/-20% window. Same readouts. Same 11-point density.")
    print("=" * 78)
    t_end = KP_STATS["C10_end"]["t"]
    early = f"(t={T_EVAL:g}s)"
    late = f"(t={t_end:g}s)"
    hdr = f"{'':>10}{'d ln C0/d ln x':>18}{'d ln C10/d ln x':>18}{'d ln C10/d ln x':>18}"
    print(hdr)
    print(f"{'x':>10}{early:>18}{early:>18}{late:>18}")
    print("-" * len(hdr))
    for name, st in (("k_on", KON_STATS), ("k_p", KP_STATS)):
        print(f"{name:>10}{st['C0']['slope']:>+18.4g}{st['C10']['slope']:>+18.4g}"
              f"{st['C10_end']['slope']:>+18.4g}")
    print()
    ratio = abs(KP_STATS["C10"]["slope"]) / max(abs(KON_STATS["C10"]["slope"]), 1e-300)
    print(f"  At t=20 s, k_p moves the activation readout {ratio:.3g}x harder than k_on does.")
    print("  Two structural facts predicted that before a single simulation ran:")
    print(f"    - k_p enters {USES['k_p']} of the {USES['k_on'] + USES['k_off'] + USES['k_p']} "
          f"parameterised rate laws; k_on enters {USES['k_on']}.")
    print("    - k_on's two substrates, P and T, are both zero at t=0.")
    print()
    print(f"  And then look at the last column. By t={KP_STATS['C10_end']['t']:g} s the k_p slope has")
    print("  collapsed toward zero: the ladder has finished filling, and where the")
    print(f"  complexes ended up is set by the survival probability ({SURVIVAL:.3f}, barely")
    print("  dependent on k_p because k_off << k_p) rather than by how fast they climbed.")
    print("  k_p controls WHEN, not HOW MUCH. A sweep that had only sampled the endpoint")
    print("  would have called k_p inert too — and been just as wrong as reading the")
    print("  k_on result as a broken pipeline.")
    print()
    print("  The transferable rule: sensitivity is a property of (parameter, readout,")
    print("  time), never of the parameter alone. Report all three or report nothing.")

## Step 6: Now look at the trajectories

**You can skim the next cell's plotting code.** What matters is what it plots, and how the two figures divide the labour:

- **Figure 1, heatmaps** — `time` on the x-axis, the swept parameter on the y-axis, colour = a species. Good for spotting *where in the (time, parameter) plane* something happens. Bad at magnitudes: each panel's colours are stretched to that panel's own range, so a flat panel and a dramatic one can look equally colourful. Always pair a heatmap with the numbers from Step 4.
- **Figure 2, overlays** — all 11 trajectories of `C10/T_Total` on **shared axes**, one panel per sweep. This is the honest comparison: same y-limits, so "eleven curves on top of each other" and "eleven curves fanned across a decade" are directly comparable.

On the storage format: BioModels outputs are arrays per DOE point, so `SweepStore` flattens the `{columns, rows, n_points, t0, t1}` payload into `time_series__columns__<i>` (the column names) plus `time_series__rows__json` (the whole table as one JSON blob), and the parser above just reads that blob back. Read it later if you ever need to write your own array-output adapter — and note the column names it prints, because the absence of a plain `time_series` column is exactly why a surrogate cannot be fitted to this table as it stands (see the checkpoint).

In [ ]:
import matplotlib.pyplot as plt

_path, _rows = newest_sweep_rows(KON_STORE)
if _rows:
    print("sweep_rows.csv columns:", ", ".join(sorted(_rows[0].keys())))
    print()

PANELS = []
if KON_OK:
    PANELS.append(("k_on", KON_STORE, "k_on (model units; SBML declares 1/s)"))
if KP_OK:
    PANELS.append(("k_p", KP_STORE, "k_p (1/s)"))

if not PANELS:
    print("No successful BioModels sweep with plottable time-series found yet.")
    print("Run Steps 2 and 5 once libroadrunner (and a cached SBML) are available.")
else:
    # ---- Figure 1: heatmaps, one row per sweep ----
    fig, axes = plt.subplots(len(PANELS), 2, figsize=(12, 4.0 * len(PANELS)), squeeze=False)
    for r_i, (var, store, ylabel) in enumerate(PANELS):
        _, recs = load_sweep(store, var)
        n = min(len(rec[1]) for rec in recs)
        t = recs[0][1][:n]
        xs = np.array([rec[0] for rec in recs])
        for c_i, (field, title, cbar) in enumerate((
            ("C0", "C0 - unphosphorylated complex (falls from 30000)",
             "C0 (molecules/cell)"),
            ("C10", "C10/T_Total - fully proofread fraction (rises from 0)",
             "C10 / T_Total"),
        )):
            scale = 1.0 if field == "C0" else 1.0 / T_TOTAL
            heat = np.vstack([rec[2][field][:n] * scale for rec in recs])
            ax = axes[r_i][c_i]
            im = ax.imshow(
                heat, origin="lower", aspect="auto",
                extent=[float(t[0]), float(t[-1]), float(xs[0]), float(xs[-1])],
                cmap="viridis",
            )
            fig.colorbar(im, ax=ax, label=cbar)
            ax.set_title(f"{var} sweep - {title}", fontsize=9)
            ax.set_xlabel("time (s)")
            ax.set_ylabel(ylabel, fontsize=8)
    fig.suptitle("Dense BioModels sweeps: each panel is auto-scaled to its OWN range", fontsize=10)
    fig.tight_layout()
    plt.show()

    # ---- Figure 2: overlays on shared axes — the honest comparison ----
    fig2, axes2 = plt.subplots(1, len(PANELS), figsize=(6.0 * len(PANELS), 4.2), squeeze=False)
    ymax = 0.0
    for var, store, _ in PANELS:
        _, recs = load_sweep(store, var)
        ymax = max(ymax, max(rec[2]["C10"].max() for rec in recs) / T_TOTAL)
    for c_i, (var, store, ylabel) in enumerate(PANELS):
        _, recs = load_sweep(store, var)
        ax = axes2[0][c_i]
        cmap = plt.get_cmap("viridis")
        lo, hi = recs[0][0], recs[-1][0]
        for x, t, s in recs:
            ax.plot(t, s["C10"] / T_TOTAL, color=cmap((x - lo) / (hi - lo)), lw=1.4)
        ax.set_ylim(0, ymax * 1.05)
        ax.set_xlabel("time (s)")
        ax.set_ylabel("C10 / T_Total")
        stats = KON_STATS if var == "k_on" else KP_STATS
        subtitle = (f"d ln C10/d ln {var} = {stats['C10']['slope']:+.3g} at t={T_EVAL:g}s"
                    if stats else "(no sensitivity summary available)")
        ax.set_title(f"{len(recs)} trajectories, {var} +/-20%\n{subtitle}", fontsize=9)
        ax.grid(alpha=0.25)
    fig2.suptitle("Same y-axis on both panels - this is the comparison the heatmaps cannot make",
                  fontsize=10)
    fig2.tight_layout()
    plt.show()

## Scientific checkpoint

Answer from the numbers you printed, not from intuition.

1. **The `k_on` sweep is a null result, and you should be able to defend it.** Name the two structural facts from the "read the model" cell that predicted it — how many of the 22 rate laws contain `k_on`, and what `P` and `T` are at `t = 0`. Then name the evidence that distinguishes *this* flat result from a broken pipeline. (Hint: what would the log-log slope have been if `k_on` never reached the simulator, and what did it actually print? Note that this is the opposite of the troubleshooting table's "outputs are all identical" symptom — the outputs are *not* identical, they are identical to five decimal places, and the difference between those two statements is the whole lesson.)

2. **`k_p` controls when, not how much.** Its slope on `C10` is large early and collapses by the end of the window. Explain that using the survival expression printed in Step 1: if `k_off ≪ k_p`, almost every complex reaches the top rung eventually, so ±20% on `k_p` changes the *arrival time* far more than the *final level*. What would have to be true of `k_off` for `k_p` to matter at the endpoint too?

3. **What you did not measure.** The figures show `C0` and `C10`. The SBML's own readout `Activation_Normalised` is an **assignment-rule variable**, and `biomodels_worker.py` builds its selection list as `["time", *rr.model.getFloatingSpeciesIds()]` — floating *species* only — so the model's headline output never reaches `sweep_rows.csv` at all. We reconstructed it as `C10 / T_Total`. Widening `selections` in `src/bayesian_metamodeling/adapters/biomodels_worker.py` is the one-line change that would record it directly. **Adapters decide what is observable; that is a modeling choice wearing plumbing's clothes.**

4. **What a surrogate would need.** Look at the column names printed above. There is a `time_series__rows__json` but no plain `time_series` column — the entire trajectory sits in one JSON blob per row. A surrogate needs *one scalar per row*, so before you could fit one to this model you would first have to pick a readout: peak time, area under the curve, `C10` at `t = 20`. That choice is a scientific commitment, not a formatting step, and it is why T5 fits its surrogate to the toy sweep, whose `y__0`/`y__1` columns are already scalar. When you get to T5, notice that `SurrogateSpec.summary_config` is exactly where this decision is supposed to live.

5. **When this whole approach is the wrong tool.** Sweeping is worth its cost when the readout responds to the parameter over the range and at the time you care about. You just measured a case where it does not — and no amount of surrogate modeling, coupling, or joint sampling downstream can recover information the sweep did not contain. When the slope comes back that close to zero, the fix is a different parameter, a different readout, or a different time; never a fancier estimator.

## Troubleshooting

| Symptom | Cause | Fix |
|---|---|---|
| `PREFLIGHT: BioModels SBML simulator missing` | `libroadrunner`/`tellurium` absent from this kernel | `pip install libroadrunner tellurium` — **PyPI-only**, not on conda-forge. Then restart the kernel. |
| Sweep skipped, but you installed roadrunner | Installed into a *different* env than the kernel | Compare the preflight's `Kernel env path` against where pip installed. They must match. |
| Hangs or fails fetching the SBML | No network, or BioModels is slow | Set `MM_BIOMODELS_OFFLINE=1` to use the cached copy and fail loudly instead of hanging. |
| Every DOE point failed | The worker subprocess ran in a different interpreter than the kernel | This was a real bug (fixed in `28835b8`). If it recurs, check `sweep_logs.jsonl` — per-point errors are recorded there, not in the summary line. |
| CLI exits 1 but rows look fine | `bayesmm run` returns 1 if *any* point failed, however many succeeded | Do what Step 3 does: count `status == 'success'` rows in `sweep_rows.csv`. The exit code is a flag, not a measurement. |
| Outputs look identical across the sweep | Either the parameter isn't reaching the model, **or** the readout genuinely doesn't depend on it | These are different, and the log-log slope tells them apart: exactly `0.0` (or `nan`) means not wired; small-but-nonzero and monotone across a dense grid means measured. Confirm the input column varies in `sweep_rows.csv`, then read the slope. |

**The habit worth taking from this table:** the summary line is not the source of
truth. `sweep_logs.jsonl` holds the per-point errors, and a sweep can report a
tidy summary while every point underneath it failed — that exact false-success is
what prompted this tutorial's rewrite. The same habit applies one level up: a
figure is not the source of truth about a *scientific* claim either, which is why
Step 4 prints numbers before Step 6 draws pictures.

## Final check: the sweeps ran, and they measured something

Two-tier, depending on whether libroadrunner was installed (preflight, Step 1).

**If the preflight passed**, the assertions below are deliberately hard to satisfy by accident:

- *structural* — each sweep produced exactly as many rows as its design has points, every row succeeded, and each row's time-series has exactly `io_schema.time_grid.n_points` samples;
- *scientific* — `k_on`'s log-log slope on `C0` is **nonzero but small** (a two-sided claim: a parameter that never reached the simulator would give exactly zero, and one that mattered would give a big number); `k_p`'s slope on `C10` at `t = 20 s` is **large**; `k_p`'s slope **collapses** by the end of the window; and the observation window is long enough for `C10` to actually fill.

A notebook that executed cleanly while teaching nothing — a sweep whose parameter never reached the model, a window too short to see the cascade, a contrast sweep that came out as flat as the null one — fails here.

**If the preflight skipped the sweeps**, the check records that instead. The `Deep CI` job sets `REQUIRE_TUTORIAL_BACKENDS=1`, which turns that skip into a failure, so a green scheduled run means the science above really ran.

In [ ]:
# Self-check: T2's BioModels sweeps produced data AND measured something (or skipped per preflight).
import csv as _csv
import json as _json
import math as _math

if not RUN_WILL_EXECUTE:
    print(f"\n[T2 self-check OK] sweeps skipped per preflight (LIBROADRUNNER_AVAILABLE={LIBROADRUNNER_AVAILABLE}, "
          f"OFFLINE={OFFLINE}, CACHE_READY={CACHE_READY}).")
else:
    def _load(store_root, spec, var):
        """Parse the newest sweep independently of the notebook's helpers."""
        _files = sorted(
            (root / store_root / "sweeps").glob("*/sweep_rows.csv"),
            key=lambda p: p.stat().st_mtime,
        )
        assert _files, f"No sweep_rows.csv under {store_root} — the sweep never ran end to end."
        with open(_files[-1], newline="") as _f:
            _rows = list(_csv.DictReader(_f))
        _planned = len(spec["design"]["grid"][var])
        _n_time = spec["io_schema"]["time_grid"]["n_points"]
        assert len(_rows) == _planned, (
            f"{_files[-1]}: {len(_rows)} rows for {_planned} planned design points."
        )
        _ok = [r for r in _rows if r.get("status") == "success"]
        assert len(_ok) == _planned, (
            f"{len(_ok)}/{_planned} points succeeded in {_files[-1]}. A partial sweep is not a "
            "pass: check sweep_logs.jsonl for the per-point errors."
        )
        _series = []
        for r in _ok:
            _tab = _json.loads(r["time_series__rows__json"])
            assert len(_tab) == _n_time, (
                f"time-series has {len(_tab)} samples, spec asked for {_n_time} in {_files[-1]}."
            )
            _series.append((float(r[var]), _tab))
        _series.sort(key=lambda item: item[0])
        return _files[-1], _series, _n_time

    def _slope(series, field, idx):
        _x0, _x1 = series[0][0], series[-1][0]
        _y0 = series[0][1][idx][field]
        _y1 = series[-1][1][idx][field]
        return _math.log(_y1 / _y0) / _math.log(_x1 / _x0)

    _kon_csv, _kon, _nt = _load(KON_STORE, SPEC_KON, "k_on")
    _kp_csv, _kp, _ = _load(KP_STORE, SPEC_KP, "k_p")

    _times = [e["time"] for e in _kon[0][1]]
    _i20 = min(range(len(_times)), key=lambda i: abs(_times[i] - 20.0))
    _iend = len(_times) - 1

    # --- science, not plumbing ---------------------------------------------
    # (a) k_on IS wired to the model (slope strictly nonzero) but the readout barely
    #     responds (slope tiny). Both halves can fail: an unwired parameter gives 0.0,
    #     a responsive one gives O(1). Measured ~0.041; bounds leave ~40x either way.
    _s_kon = _slope(_kon, "C0", _i20)
    assert 1e-3 < abs(_s_kon) < 0.5, (
        f"d ln C0 / d ln k_on = {_s_kon:.3e} at t={_times[_i20]:g}s. Expected a small but "
        "clearly nonzero slope: ~0 means k_on never reached the simulator (check the "
        "adapter input_mapping), large means the model or the sweep range changed and "
        "the null-result lesson in Step 4 no longer holds."
    )
    # (b) k_p moves the activation readout hard at t=20 s. Measured ~5.7; floor 2.0.
    _s_kp = _slope(_kp, "C10", _i20)
    assert _s_kp > 2.0, (
        f"d ln C10 / d ln k_p = {_s_kp:.3f} at t={_times[_i20]:g}s, expected > 2. The contrast "
        "sweep is supposed to be the one that responds; without it Step 5 teaches nothing."
    )
    # (c) The contrast is the lesson: k_p must dominate k_on by orders of magnitude.
    _s_kon_c10 = _slope(_kon, "C10", _i20)
    assert abs(_s_kp) > 1000 * abs(_s_kon_c10), (
        f"k_p slope {_s_kp:.3g} vs k_on slope {_s_kon_c10:.3g} on C10 — expected >1000x."
    )
    # (d) Sensitivity is a property of (parameter, readout, TIME): the k_p slope must
    #     collapse once the ladder has filled. Measured ~-0.003; bound 0.5.
    _s_kp_end = _slope(_kp, "C10", _iend)
    assert abs(_s_kp_end) < 0.5, (
        f"d ln C10 / d ln k_p = {_s_kp_end:.3f} at t={_times[_iend]:g}s, expected near zero. "
        "If this is large the observation window no longer reaches the plateau and the "
        "'k_p controls when, not how much' conclusion is unsupported."
    )
    # (e) The window must be long enough to see the cascade fill at all.
    _peak = max(e["C10"] for _, tab in _kon for e in tab) / T_TOTAL
    assert _peak > 0.5, (
        f"C10 peaks at only {_peak:.3f} x T_Total inside the window — too short to show the "
        "ladder filling, so Step 5's endpoint conclusion has nothing to stand on."
    )

    print(f"\n[T2 self-check OK] k_on: {len(_kon)}/{len(_kon)} points, {_nt} time samples, "
          f"d ln C0/d ln k_on = {_s_kon:+.4f}, d ln C10/d ln k_on = {_s_kon_c10:+.2e}")
    print(f"[T2 self-check OK] k_p : {len(_kp)}/{len(_kp)} points, "
          f"d ln C10/d ln k_p = {_s_kp:+.4f} at t={_times[_i20]:g}s -> "
          f"{_s_kp_end:+.4f} at t={_times[_iend]:g}s; C10 peaks at {_peak:.3f} x T_Total")